# Ecosistema de procesamiento paralelo para datos masivos

## Módulo 3. Lección 1: El paradigma de programación MapReduce: hadoop-streaming

### Ejercicios resueltos 02

**IMPORTANTE**

Ejecuta las celdas en orden para evitar errores de ejecución. Esto garantiza que todos los archivos necesarios se creen y se ubiquen correctamente a lo largo del proceso.

#### **Obtención de los datos**

In [7]:
# creamos la carpeta para guardar los ficheros
! mkdir -p /media/notebooks/futbol

# Descargamos el fichero penaltis y resultados
! wget https://github.com/Vega90/datasets/raw/main/resultados.csv \
-O /media/notebooks/futbol/resultados.csv

! wget https://github.com/Vega90/datasets/raw/main/penaltis.csv \
-O /media/notebooks/futbol/penaltis.csv

--2024-08-28 11:42:49--  https://github.com/Vega90/datasets/raw/main/resultados.csv
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/Vega90/datasets/main/resultados.csv [following]
--2024-08-28 11:42:49--  https://raw.githubusercontent.com/Vega90/datasets/main/resultados.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 711486 (695K) [text/plain]
Saving to: ‘/media/notebooks/futbol/resultados.csv’

/media/notebooks/fu 100%[===================>] 694.81K  --.-KB/s    in 0.07s   

2024-08-28 11:42:50 (9.83 MB/s) - ‘/media/notebooks/futbol/resultados.csv’ saved [711486/711486]

--2024-08-28 11:42:5

In [1]:
# subimos los ficheros a HDFS 
! hdfs dfs -mkdir /futbol
! hdfs dfs -put /media/notebooks/futbol/resultados.csv /futbol
! hdfs dfs -put /media/notebooks/futbol/penaltis.csv /futbol
! hdfs dfs -ls /futbol

put: `/media/notebooks/futbol/resultados.csv': No such file or directory
put: `/media/notebooks/futbol/penaltis.csv': No such file or directory


#### **Creación carpeta para los ejercicios**

In [2]:
import os
# creamos la carpeta para los ejercicios
! mkdir -p /media/notebooks/futbol/mapreduce
# nos vamos a esa carpeta y comprobamos donde estamos
os.chdir("/media/notebooks/futbol/mapreduce")
! pwd

/media/notebooks/futbol/mapreduce


#### **Dataframe de pandas para comprender los datos y comparar la salida**

Utilizaremos un dataframe de pandas para ayudarnos a entender los datos y también podemos comprobar los ejercicios comparando la salida dada y la salida esperada.

In [8]:
import pandas as pd

# dataframe resultados
columnas = ['date','home_team','away_team','home_score','away_score','tournament','city','country']
df_resultados = pd.read_csv('/media/notebooks/futbol/resultados.csv', names=columnas)

# dataframe penaltis
columnas = ['date','home_team','away_team','winner']
df_penaltis = pd.read_csv('/media/notebooks/futbol/penaltis.csv', names=columnas)

# mostramos los primeros registros de resultados
df_resultados.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country
0,1956-09-23,Germany,Netherlands,2,1,Friendly,Essen,Germany
1,1957-07-28,Germany,England,1,1,Friendly,Stuttgart,Germany
2,1957-10-13,Germany,Netherlands,2,0,Friendly,Berlin,Germany
3,1957-11-03,Netherlands,Austria,8,1,European Championship,Berlin,Germany
4,1957-11-03,Germany,England,0,4,European Championship,Berlin,Germany


#### **Ejercicio 05.** Filtrado de registros. Partidos de España.

El primer ejercicio consiste en contar el número de valores únicos de la columna county. De esa manera, sabremos exactamente de cuántos condados tenemos datos sobre los casos de Coronavirus que vamos a estudiar.

**Proceso MAP 05**

In [20]:
%%writefile 05_mapper.py
#!/usr/bin/env python3
import sys

# funcion aux para determinar el resultado
def obtener_resultado(equipo, goles_locales, goles_visitantes):
    if goles_locales == goles_visitantes:
        return None  # hay empate
    if goles_locales > goles_visitantes:
        return 'victoria'
    return 'derrota'
        
# leemos cada linea de entrada
for linea in sys.stdin:
    # quitamos espacios y dividimos
    campos = linea.strip().split(',')

    # nos quedamos con los campos que nos interesan
    fecha, local, visitante = campos[0], campos[1], campos[2]
    goles_locales, goles_visitantes = int(campos[3]), int(campos[4])

    # espanya juega como local
    if local.lower() == 'spain':
        resultado = obtener_resultado(local, goles_locales, goles_visitantes)
        if resultado:
            print(f"{resultado}\t{fecha},{visitante},{goles_locales}-{goles_visitantes}")

    # espanya juega como visitante
    elif visitante.lower() == 'spain':
        resultado = obtener_resultado(visitante, goles_visitantes, goles_locales)
        if resultado:
            print(f"{resultado}\t{fecha},{local},{goles_locales}-{goles_visitantes}")


Overwriting 05_mapper.py


**Proceso REDUCE 05**

In [41]:
%%writefile 05_reducer.py
#!/usr/bin/env python3
import sys

# funcion para imprimir los resultados
def imprimir_resultados(lista, tipo):
    print(f"\n{tipo.upper()}:")
    for item in lista:
        fecha, oponente, goles = item.split(',')
        print(f"  Fecha: {fecha}\tResultado: {goles}\tOponente: {oponente}")
        
# listas para victorias y derrotas
victorias = []
derrotas = []

# procesamos cada linea
for linea in sys.stdin:
    resultado, detalles = linea.strip().split('\t')
    # Agregamos los detalles a la lista correspondiente
    (victorias if resultado == 'victoria' else derrotas).append(detalles)

# numero de derrotas y victorias
print(f"España ha ganado {len(victorias)} veces:")
print(f"España ha perdido {len(derrotas)} veces:")

# imprimimos las listas de victorias y derrotas
imprimir_resultados(victorias, "victorias")
imprimir_resultados(derrotas, "derrotas")

Overwriting 05_reducer.py


In [42]:
# ejecutamos nuestro map-reduce y vemos el resultado desde la consola 
! cat /media/notebooks/futbol/resultados.csv \
| python3 05_mapper.py | sort | python3 05_reducer.py > salida-05

! cat salida-05

España ha ganado 125 veces:
España ha perdido 59 veces:

VICTORIAS:
  Fecha: 1986-06-10	Resultado: 0-3	Oponente: Switzerland
  Fecha: 1987-12-19	Resultado: 1-0	Oponente: Belgium
  Fecha: 1988-03-20	Resultado: 1-0	Oponente: Bulgaria
  Fecha: 1992-03-22	Resultado: 0-1	Oponente: Republic of Ireland
  Fecha: 1994-03-20	Resultado: 17-0	Oponente: Slovenia
  Fecha: 1994-04-24	Resultado: 4-0	Oponente: Belgium
  Fecha: 1994-05-29	Resultado: 0-8	Oponente: Slovenia
  Fecha: 1996-03-31	Resultado: 5-1	Oponente: Romania
  Fecha: 1996-09-08	Resultado: 2-1	Oponente: England
  Fecha: 1997-07-05	Resultado: 0-1	Oponente: Russia
  Fecha: 2000-04-01	Resultado: 1-2	Oponente: Netherlands
  Fecha: 2003-03-26	Resultado: 0-1	Oponente: Netherlands
  Fecha: 2004-02-29	Resultado: 9-1	Oponente: Belgium
  Fecha: 2007-05-30	Resultado: 0-3	Oponente: Belarus
  Fecha: 2008-02-16	Resultado: 4-0	Oponente: Northern Ireland
  Fecha: 2008-05-03	Resultado: 6-1	Oponente: Belarus
  Fecha: 2008-05-08	Resultado: 4-1	Oponente: Cze

**Ejecución Hadoop Streaming 05**

In [9]:
! hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-files 05_mapper.py,05_reducer.py \
-mapper 05_mapper.py \
-reducer 05_reducer.py \
-input /futbol/resultados.csv \
-output /futbol/mapreduce/05_salida

packageJobJar: [/tmp/hadoop-unjar2018823454391409366/] [] /tmp/streamjob7637070950287095750.jar tmpDir=null
2024-08-16 10:39:58,799 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.7:8032
2024-08-16 10:39:59,096 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.7:8032
2024-08-16 10:39:59,619 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1723804082756_0002
2024-08-16 10:40:00,704 INFO mapred.FileInputFormat: Total input files to process : 1
2024-08-16 10:40:01,149 INFO mapreduce.JobSubmitter: number of splits:2
2024-08-16 10:40:01,561 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1723804082756_0002
2024-08-16 10:40:01,562 INFO mapreduce.JobSubmitter: Executing with tokens: []
2024-08-16 10:40:02,189 INFO conf.Configuration: resource-types.xml not found
2024-08-16 10:40:02,190 INFO resource.ResourceUtils

In [10]:
# Visualizamos la salida del ejercicio 5
! hdfs dfs -cat /futbol/mapreduce/05_salida/part-00000

España ha ganado 125 veces:	
España ha perdido 59 veces:	
	
VICTORIAS:	
  Fecha: 2024-07-16	Resultado: 2-0	Oponente: Belgium
  Fecha: 2024-06-04	Resultado: 3-2	Oponente: Denmark
  Fecha: 2024-05-31	Resultado: 0-2	Oponente: Denmark
  Fecha: 2024-04-09	Resultado: 3-1	Oponente: Czech Republic
  Fecha: 2024-04-05	Resultado: 0-7	Oponente: Belgium
  Fecha: 2024-02-28	Resultado: 2-0	Oponente: France
  Fecha: 2024-02-23	Resultado: 3-0	Oponente: Netherlands
  Fecha: 2023-12-05	Resultado: 5-3	Oponente: Sweden
  Fecha: 2023-10-31	Resultado: 1-7	Oponente: Switzerland
  Fecha: 2023-10-27	Resultado: 0-1	Oponente: Italy
  Fecha: 2023-09-26	Resultado: 5-0	Oponente: Switzerland
  Fecha: 2023-09-22	Resultado: 2-3	Oponente: Sweden
  Fecha: 2023-08-20	Resultado: 1-0	Oponente: England
  Fecha: 2023-08-15	Resultado: 2-1	Oponente: Sweden
  Fecha: 2023-08-11	Resultado: 2-1	Oponente: Netherlands
  Fecha: 2023-08-05	Resultado: 1-5	Oponente: Switzerland
  Fecha: 2023-07-26	Resultado: 5-0	Oponente: Zambia
  Fecha

**Comprobacion en el dataframe**

Esta vez necesitamos algunas operaciones auxiliares. Vamos a crear un dataframe nuevo llamado df_spain, una nueva columna con el ganador y a partir de ahí buscaremos el número de derrotas y de victorias de España.

In [9]:
import warnings
warnings.filterwarnings('ignore')

# filtramos las filas donde juega españa
dt_spain = df_resultados.loc[(df_resultados['home_team']=='Spain') | (df_resultados['away_team']=='Spain')]

# creamos una columna con el nombre del ganador (eliminamos los empates)
dt_spain['ganador'] = dt_spain['home_team']
dt_spain.loc[dt_spain['away_score'] > dt_spain['home_score'], 'ganador'] = dt_spain['away_team']
dt_spain.loc[dt_spain['home_score'] == dt_spain['away_score'], 'ganador'] = pd.NA
dt_spain = dt_spain.dropna()

In [10]:
# extraemos las victorias de españa y el número de victorias
dt_victorias = dt_spain.loc[dt_spain['ganador'] == 'Spain'][['date', 'ganador', 'home_team','away_team','home_score','away_score']]
num_victorias = dt_victorias.shape[0]
print(f"Nº de victorias en el dataFrame: {num_victorias}")
# mostramos las cinco primeras
dt_victorias.sort_values(by='date', ascending=False).head()

Nº de victorias en el dataFrame: 125


,date,ganador,home_team,away_team,home_score,away_score
9627,2024-07-16,Spain,Spain,Belgium,2,0
9553,2024-06-04,Spain,Spain,Denmark,3,2
9499,2024-05-31,Spain,Denmark,Spain,0,2
9459,2024-04-09,Spain,Spain,Czech Republic,3,1
9412,2024-04-05,Spain,Belgium,Spain,0,7


In [13]:
dt_derrotas = dt_spain.loc[dt_spain['ganador'] != 'Spain'][['date', 'ganador', 'home_team','away_team','home_score','away_score']]
num_derrotas = dt_derrotas.shape[0]
print(f"Nº de derrotas en el dataFrame: {num_derrotas}")
dt_derrotas.sort_values(by='date', ascending=False).head()

Nº de derrotas en el dataFrame: 59


,date,ganador,home_team,away_team,home_score,away_score
9593,2024-07-12,Czech Republic,Czech Republic,Spain,2,1
9226,2023-12-01,Italy,Spain,Italy,2,3
8904,2023-07-31,Japan,Japan,Spain,4,0
8654,2023-02-19,Australia,Australia,Spain,3,2
8421,2022-07-20,England,England,Spain,2,1


Algunos registros no aparecen en el mismo orden que en el proceso MapReduce, pero están todos los registros. 

#### **Ejercicio 06.** Top N. Los cinco mejores equipos del mundo.

Este ejercicio nos servirá para extraer los tres equipos mejores del mundo. Consideraremos aquellos que han ganado más competiciones oficiales, por tanto, los partidos "amistosos" debemos desecharlos de nuestros resultados. Para ello, debemos fijarnos en el campo 'tournament', que corresponde al motivo por el cuál se disputó el partido.

Además de ello, consideraremos también los países que han jugado más de 100 partidos. Esto lo haremos para tener una visión más general. Podría ocurrir que un país haya disputado un único partido y lo haya ganado. Su puntuación promedio sería del 100%, pero no sería realista.

En la fase Map tendremos que obtener los campos que nos interesan. En este caso los equipos, los resultados y la competición. Utilizaremos una puntuación para cada equipo según haya ganado (+1), empatado (0) o perdido(-1). Así en la fase reduce obtendremos las puntuaciones totales.

**Proceso MAP 06**

In [17]:
%%writefile 06_mapper.py
#!/usr/bin/env python3
import sys

# entrada estandar STDIN
for linea in sys.stdin:
    # eliminamos espacios y dividimos
    campos = linea.strip().split(',')

    # nos quedamos con los campos que queremos
    equipo1, equipo2 = campos[1], campos[2]
    goles_locales, goles_visitantes = int(campos[3]), int(campos[4])
    competicion = campos[5]

    # inicializamos resultados
    resultado1, resultado2 = 0, 0

    # eliminamos los amistosos
    if competicion != 'Friendly':
        # calculamos los resultados basados en los goles
        if goles_locales == goles_visitantes:
            resultado1, resultado2 = 0, 0  # empate
        elif goles_locales > goles_visitantes:
            resultado1, resultado2 = 1, -1  # victoria para el primer equipo
        else:
            resultado1, resultado2 = -1, 1  # victoria para el segundo equipo

        # Imprimimos los resultados con conteo
        print(f"{equipo1}\t{resultado1}\t1")
        print(f"{equipo2}\t{resultado2}\t1")

Overwriting 06_mapper.py


**Proceso Reduce 06**

In [45]:
%%writefile 06_reducer.py
#!/usr/bin/env python3
import sys
from collections import defaultdict

# utilizamos un diccionario para almacenar equipos y sus estadisticas
# [puntuación_total, puntuación_sumada, conteo_total]
equipos = defaultdict(lambda: [0, 0, 0])

# procesamos cada linea de entrada
for linea in sys.stdin:
    equipo, puntuacion, conteo = linea.strip().split('\t')
    puntuacion, conteo = int(puntuacion), int(conteo)
    datos = equipos[equipo]
    datos[1] += puntuacion
    datos[2] += conteo
    promedio = datos[1] / datos[2]
    datos[0] = round(promedio * 100, 3)

# filtramos los equipos que han jugado más de 100 partidos
equipos_filtrados = {k: v for k, v in equipos.items() if v[2] > 100}

# ordenamos y obtenemos el top 5 equipos por puntuacion promedio
top_5_equipos = sorted(equipos_filtrados.items(), key=lambda x: x[1][0], reverse=True)[:5]

# imprimimos los resultados
print("\nLos cinco mejores equipos con más de 100 partidos:")
for equipo, datos in top_5_equipos:
    print(f"\n   - {equipo}:")
    print(f"\tPuntuación Promedio = {datos[0]}%")
    print(f"\tPuntuación total = {datos[1]}")
    print(f"\tPartidos jugados = {datos[2]}")

Overwriting 06_reducer.py


In [46]:
# ejecutamos nuestro map-reduce desde la consola
! cat /media/notebooks/futbol/resultados.csv \
| python3 06_mapper.py | sort | python3 06_reducer.py > salida-06

! cat salida-06


Los cinco mejores equipos con más de 100 partidos:

   - United States:
	Puntuación Promedio = 65.642%
	Puntuación total = 235
	Partidos jugados = 358

   - Germany:
	Puntuación Promedio = 52.819%
	Puntuación total = 178
	Partidos jugados = 337

   - North Korea:
	Puntuación Promedio = 43.716%
	Puntuación total = 80
	Partidos jugados = 183

   - Brazil:
	Puntuación Promedio = 41.026%
	Puntuación total = 96
	Partidos jugados = 234

   - France:
	Puntuación Promedio = 39.259%
	Puntuación total = 106
	Partidos jugados = 270


**Ejecución Hadoop Streaming 06**

In [23]:
! hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-files 06_mapper.py,06_reducer.py \
-mapper 06_mapper.py \
-reducer 06_reducer.py \
-input /futbol/resultados.csv \
-output /futbol/mapreduce/06_salida

packageJobJar: [/tmp/hadoop-unjar7102918890620171140/] [] /tmp/streamjob3537454056743600565.jar tmpDir=null
2024-08-16 10:51:13,057 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.7:8032
2024-08-16 10:51:13,353 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.7:8032
2024-08-16 10:51:13,831 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1723804082756_0003
2024-08-16 10:51:14,962 INFO mapred.FileInputFormat: Total input files to process : 1
2024-08-16 10:51:15,338 INFO mapreduce.JobSubmitter: number of splits:2
2024-08-16 10:51:15,799 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1723804082756_0003
2024-08-16 10:51:15,799 INFO mapreduce.JobSubmitter: Executing with tokens: []
2024-08-16 10:51:16,232 INFO conf.Configuration: resource-types.xml not found
2024-08-16 10:51:16,232 INFO resource.ResourceUtils

In [24]:
# Visualizamos la salida del ejercicio 6
! hdfs dfs -cat /futbol/mapreduce/06_salida/part-00000

	
Los cinco mejores equipos con más de 100 partidos:	
	
   - United States:	
	Puntuación Promedio = 65.642%, 
	Puntuación total = 235, 
	Partidos jugados = 358
	
   - Germany:	
	Puntuación Promedio = 52.819%, 
	Puntuación total = 178, 
	Partidos jugados = 337
	
   - North Korea:	
	Puntuación Promedio = 43.716%, 
	Puntuación total = 80, 
	Partidos jugados = 183
	
   - Brazil:	
	Puntuación Promedio = 41.026%, 
	Puntuación total = 96, 
	Partidos jugados = 234
	
   - France:	
	Puntuación Promedio = 39.259%, 
	Puntuación total = 106, 
	Partidos jugados = 270


In [26]:
# funcion para calcular los puntos
def calcular_puntos(fila):
    if fila['home_score'] > fila['away_score']:
        return 1, -1
    elif fila['away_score'] > fila['home_score']:
        return -1, 1
    else:
        return 0, 0

In [27]:
import warnings
warnings.filterwarnings('ignore')

#filtramos que no sean amistosos
df_puntos = df_resultados.loc[df_resultados['tournament']!='Friendly']

# creamos las columnas para las puntuaciones de ambos equipos
df_puntos['puntos_local'], df_puntos['puntos_visitante'] = zip(*df_puntos.apply(calcular_puntos, axis=1))

# Acumulamos los resultados para cada equipo
equipos = pd.concat([
    df_puntos[['home_team', 'puntos_local']].rename(columns={'home_team': 'equipo', 'puntos_local': 'puntos'}),
    df_puntos[['away_team', 'puntos_visitante']].rename(columns={'away_team': 'equipo', 'puntos_visitante': 'puntos'})
])

# creamos un dataframe con los puntos y los partidos totales
df_puntuaciones = equipos.groupby('equipo').agg(
                puntuacion_total=('puntos', 'sum'), partidos_jugados=('puntos', 'count')
                ).reset_index()

# calculamos la puntuación promedio
df_puntuaciones['puntuacion_promedio'] = df_puntuaciones['puntuacion_total'] / df_puntuaciones['partidos_jugados']
df_puntuaciones['puntuacion_promedio'] = round(df_puntuaciones['puntuacion_promedio']*100,3)

# filtramos para que hayan jugado más de 100 partidos
df_puntuaciones = df_puntuaciones.loc[df_puntuaciones['partidos_jugados'] > 100]

# ordenamos y nos quedamos con los cinco primeros
df_puntuaciones.sort_values(by='puntuacion_promedio', ascending=False).head(5)

,equipo,puntuacion_total,partidos_jugados,puntuacion_promedio
224,United States,235,358,65.642
77,Germany,178,337,52.819
154,North Korea,80,183,43.716
28,Brazil,96,234,41.026
73,France,106,270,39.259


#### **Ejercicio 07.** Unión de datasets 01.

En este ejercicio Nº 7 vamos a realizar la unión de dos datasets mediante hadoop streaming. En concreto, vamos a utilizar el dataset penaltis y el dataset resultados, y vamos a unirlos en un nuevo dataset al que llamaremos combinado.

Para realizar la unión nos fijaremos en el campo común a ambos datasets. En este caso, ambos data sets comparten el campo de 'date'. Por lo tanto, la unión la realizaremos por dicho campo. En este ejercicio realizaremos un Inner Join para combinar los campos que tienen la clave común.

**Proceso MAP 07. Fase 01**

En esta fase Map tendremos que obtener los campos que nos interesan. El campo 'clave' es la fecha en la que se jugó el partido, por tanto, ese campo debe ser el primero que escribamos con la fase MAP. Posteriormente, añadiremos un prefijo (dt1 o dt2) antes del resto de valores que enviaremos a la siguiente fase.

Para el dataset de penaltis solo guardaremos el ganador ya que los campos de equipo local y visitante están en ambos dataset y tendríamos campos repetidos.

In [47]:
%%writefile 07_mapper1.py
#!/usr/bin/env python3
import sys

# si es dataset1 el prefijo es dt1 si no dt2
prefijo = 'dt1' if 'dataset1' in sys.argv[1] else 'dt2'

# entrada estandar STDIN
for linea in sys.stdin:
    # eliminamos espacios y dividimos
    campos = linea.strip().split(',')

    # la clave es date-home_team-away_team
    clave = f"{campos[0]}-{campos[1]}-{campos[2]}"

    if prefijo == 'dt1':
        # solo nos interesa si hay empate
        goles_locales, goles_visitantes = int(campos[3]), int(campos[4])
        if goles_locales == goles_visitantes:
            print(f"{clave}\t{prefijo}\t{','.join(campos)}")
    else:
        # nos quedamos solo con el ganador
        ganador = campos[3]
        print(f"{clave}\t{prefijo}\t{ganador}")


Overwriting 07_mapper1.py


In [28]:
# ejecutamos desde la consola para el dataset 1
! cat /media/notebooks/futbol/resultados.csv \
| python3 07_mapper1.py dataset1 > temp-07-1

! cat temp-07-1 | head -n 5

1957-07-28-Germany-England	dt1	1957-07-28,Germany,England,1,1,Friendly,Stuttgart,Germany
1971-08-22-Denmark-Italy	dt1	1971-08-22,Denmark,Italy,1,1,World Cup,Guadalajara,Mexico
1978-07-08-Finland-Sweden	dt1	1978-07-08,Finland,Sweden,0,0,Nordic Championship,Horsens,Denmark
1979-07-08-Finland-Sweden	dt1	1979-07-08,Finland,Sweden,1,1,Nordic Championship,Oslo,Norway
1979-07-20-France-Scotland	dt1	1979-07-20,France,Scotland,0,0,Euro,Rimini,Italy
cat: write error: Broken pipe


In [29]:
# ejecutamos desde la consola para el dataset 2
! cat /media/notebooks/futbol/penaltis.csv \
| python3 07_mapper1.py dataset2 > temp-07-2

! cat temp-07-2 | head -n 5

1984-05-27-England-Sweden	dt2	Sweden
1988-06-12-China PR-Brazil	dt2	Brazil
1995-06-13-Sweden-China PR	dt2	China PR
1996-05-18-Japan-Canada	dt2	Japan
1998-07-25-China PR-Norway	dt2	China PR


**Proceso 1. Hadoop Streaming 07**

Obtendremos el fichero temporal para el dataset1, resultados.

La fase reduce no es necearia. Lo que haremos será establercer un parámetro **-D mapreduce.job.reduces=0** en el que indicaremos que no utilizaremos un trabajo reduce. Esto hará que Hadoop Streaming tome la salida del mapper como salida final y escriba sus resultados en el fichero temporal.

In [30]:
! hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-files 07_mapper1.py \
-D mapreduce.job.reduces=0 \
-mapper '07_mapper1.py dataset1' \
-input /futbol/resultados.csv \
-output /futbol/mapreduce/07_combinado_1

packageJobJar: [/tmp/hadoop-unjar7015143590009130449/] [] /tmp/streamjob8191006509956781794.jar tmpDir=null
2024-08-16 10:56:05,904 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.7:8032
2024-08-16 10:56:06,196 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.7:8032
2024-08-16 10:56:06,952 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1723804082756_0004
2024-08-16 10:56:07,752 INFO mapred.FileInputFormat: Total input files to process : 1
2024-08-16 10:56:08,031 INFO mapreduce.JobSubmitter: number of splits:2
2024-08-16 10:56:08,420 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1723804082756_0004
2024-08-16 10:56:08,420 INFO mapreduce.JobSubmitter: Executing with tokens: []
2024-08-16 10:56:08,827 INFO conf.Configuration: resource-types.xml not found
2024-08-16 10:56:08,828 INFO resource.ResourceUtils

**Proceso 2. Hadoop Streaming 07**

Obtendremos el fichero temporal para el dataset1, penaltis.

In [31]:
! hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-files 07_mapper1.py \
-D mapreduce.job.reduces=0 \
-mapper '07_mapper1.py dataset2' \
-input /futbol/penaltis.csv \
-output /futbol/mapreduce/07_combinado_2

packageJobJar: [/tmp/hadoop-unjar4656747512388626301/] [] /tmp/streamjob6924853285509552411.jar tmpDir=null
2024-08-16 10:57:49,512 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.7:8032
2024-08-16 10:57:49,822 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.7:8032
2024-08-16 10:57:50,539 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1723804082756_0005
2024-08-16 10:57:51,613 INFO mapred.FileInputFormat: Total input files to process : 1
2024-08-16 10:57:52,232 INFO mapreduce.JobSubmitter: number of splits:2
2024-08-16 10:57:52,652 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1723804082756_0005
2024-08-16 10:57:52,653 INFO mapreduce.JobSubmitter: Executing with tokens: []
2024-08-16 10:57:53,154 INFO conf.Configuration: resource-types.xml not found
2024-08-16 10:57:53,156 INFO resource.ResourceUtils

**Visualización de la salida temporal**

In [32]:
# Visualizamos la salida temporal
print('\nTemporal 1:\n')
! hdfs dfs -cat /futbol/mapreduce/07_combinado_1/part-* | head -n 5

print('\n\nTemporal 2:\n')
! hdfs dfs -cat /futbol/mapreduce/07_combinado_2/part-* | head -n 5


Temporal 1:

1957-07-28-Germany-England	dt1	1957-07-28,Germany,England,1,1,Friendly,Stuttgart,Germany
1971-08-22-Denmark-Italy	dt1	1971-08-22,Denmark,Italy,1,1,World Cup,Guadalajara,Mexico
1978-07-08-Finland-Sweden	dt1	1978-07-08,Finland,Sweden,0,0,Nordic Championship,Horsens,Denmark
1979-07-08-Finland-Sweden	dt1	1979-07-08,Finland,Sweden,1,1,Nordic Championship,Oslo,Norway
1979-07-20-France-Scotland	dt1	1979-07-20,France,Scotland,0,0,Euro,Rimini,Italy
cat: Unable to write to output stream.
cat: Unable to write to output stream.


Temporal 2:

1984-05-27-England-Sweden	dt2	Sweden
1988-06-12-China PR-Brazil	dt2	Brazil
1995-06-13-Sweden-China PR	dt2	China PR
1996-05-18-Japan-Canada	dt2	Japan
1998-07-25-China PR-Norway	dt2	China PR
cat: Unable to write to output stream.


**FASE 2. Combinación de resultados parciales.**

Ahora crearemos un fichero mapper2 que únicamente leerá el contenido de las líneas de ambos ficheros temporales y lo dejará pasar para que se encargue la fase reduce de realizar la combinación.

**Proceso MAP 07. Fase 02**

In [33]:
%%writefile 07_mapper2.py
#!/usr/bin/env python3
import sys

for linea in sys.stdin:
    print(linea.strip())

Writing 07_mapper2.py


**Proceso REDUCE 07**

In [36]:
%%writefile 07_reducer.py
#!/usr/bin/env python3
import sys

dict_dt1 = {}
dict_dt2 = {}

# por cada linea leemos la clave, el prefijo y los campos
for linea in sys.stdin:
    clave, dataset, campos = linea.strip().split('\t')
	
	# ahora la clave es unica, no puede existir duplicado
    if dataset == 'dt1':
        dict_dt1[clave] = [campos]
    elif dataset == 'dt2':
        dict_dt2[clave] = [campos]

# imprimimos los resultados si existe la clave para los dos
for clave in dict_dt1:
    if clave in dict_dt2:
        campos1 = ','.join(dict_dt1[clave])
        campos2 = ','.join(dict_dt2[clave])
        print(f"{campos1},{campos2}")

Overwriting 07_reducer.py


In [37]:
# ejecutamos nuestro map-reduce y vemos el resultado desde la consola 
! cat temp-07-1 temp-07-2 | python3 07_mapper2.py | sort > temp_sorted-7
! cat temp_sorted-7 | python3 07_reducer.py > salida-07
! cat salida-07 | head -n 5

1988-06-12,China PR,Brazil,0,0,FIFA Women's Invitation Tournament,Guangzhou,China PR,Brazil
1995-06-13,Sweden,China PR,1,1,FIFA World Cup,Helsingborg,Sweden,China PR
1996-05-18,Japan,Canada,0,0,U.S. Cup,"Washington, D.C.",United States,Japan
1998-07-25,China PR,Norway,1,1,Goodwill Games,Uniondale,United States,China PR
1998-07-27,Denmark,Norway,1,1,Goodwill Games,Uniondale,United States,Norway


**Proceso 3. Hadoop Streaming 07**

El último proceso Hadoop Streaming debe tener dos carpetas de entrada, las carpetas que contienen los ficheros temporales creados en los procesos anteriores.

In [38]:
! hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-files 07_mapper2.py,07_reducer.py \
-mapper 07_mapper2.py \
-reducer 07_reducer.py \
-input /futbol/mapreduce/07_combinado_2/part-* \
-input /futbol/mapreduce/07_combinado_1/part-* \
-output /futbol/mapreduce/07_salida

packageJobJar: [/tmp/hadoop-unjar7444175965645559981/] [] /tmp/streamjob6441117249651240966.jar tmpDir=null
2024-08-16 11:03:03,858 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.7:8032
2024-08-16 11:03:04,209 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.7:8032
2024-08-16 11:03:04,670 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1723804082756_0006
2024-08-16 11:03:05,473 INFO mapred.FileInputFormat: Total input files to process : 4
2024-08-16 11:03:05,817 INFO mapreduce.JobSubmitter: number of splits:4
2024-08-16 11:03:06,255 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1723804082756_0006
2024-08-16 11:03:06,256 INFO mapreduce.JobSubmitter: Executing with tokens: []
2024-08-16 11:03:06,761 INFO conf.Configuration: resource-types.xml not found
2024-08-16 11:03:06,762 INFO resource.ResourceUtils

In [39]:
# Visualizamos la salida del ejercicio 7
! hdfs dfs -cat /futbol/mapreduce/07_salida/part-00000 | head -n 5

1988-06-12,China PR,Brazil,0,0,FIFA Women's Invitation Tournament,Guangzhou,China PR,Brazil	
1995-06-13,Sweden,China PR,1,1,FIFA World Cup,Helsingborg,Sweden,China PR	
1996-05-18,Japan,Canada,0,0,U.S. Cup,"Washington, D.C.",United States,Japan	
1998-07-25,China PR,Norway,1,1,Goodwill Games,Uniondale,United States,China PR	
1998-07-27,Denmark,Norway,1,1,Goodwill Games,Uniondale,United States,Norway	


**Comparación entre la salida obtenida y la esperada**

In [40]:
# dataframe con los empates
df_empates = df_resultados.loc[df_resultados['home_score'] == df_resultados['away_score']]

# realizamos la union
df_combinado = pd.merge(df_empates, df_penaltis, on=['date','home_team','away_team']).dropna()
df_combinado.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,winner
0,1988-06-12,China PR,Brazil,0,0,FIFA Women's Invitation Tournament,Guangzhou,China PR,Brazil
1,1995-06-13,Sweden,China PR,1,1,FIFA World Cup,Helsingborg,Sweden,China PR
2,1996-05-18,Japan,Canada,0,0,U.S. Cup,"Washington, D.C.",United States,Japan
3,1998-07-25,China PR,Norway,1,1,Goodwill Games,Uniondale,United States,China PR
4,1998-07-27,Denmark,Norway,1,1,Goodwill Games,Uniondale,United States,Norway
